# NINAAD WAGLE | BTECH AI SEM V | I065 B2 | NLP LAB 6 (PT2)

# Rule based NER, written from scratch

Part 1 built a tagger that **learnt** from the labelled column - it counted which tag each
word was given and remembered the answer. This part does the same job using **rules only**.

The rules are written as if nothing is known about this dataset in advance:

- **No NLP library.** No NLTK, no spaCy. Only `pandas` to read the file, and one scoring
  function at the very end.
- **No tags from the dataset.** The `Tag` and `POS` columns are not read while the rules are
  written. `Tag` is opened once at the end, only to check the answers.
- **No lists of names.** No list of countries, no list of nationalities, no list of titles.
  Writing such a list would be smuggling in knowledge the rules are supposed to work out for
  themselves.
- **No train / test split.** There is nothing to train, so nothing is held back. The rules run
  over all 47,959 sentences.

So the rules are left with only what the text itself shows - **capital letters, numbers,
where a word sits in the sentence, and the words on either side of it.**

Four kinds of entity are looked for: **person**, **organisation**, **place** and **date**.

# Task c : Applying NER on the dataset

## i : Reading the corpus as plain sentences

The file is read as in Part 1 - the sentence number is filled downwards and the ten empty rows
are dropped. `sentences` keeps only the words, and that is all the rules may look at. `gold`
keeps the real tags to one side, untouched until the checking step.

In [1]:
import os
import pandas as pd

path = "/kaggle/input/entity-annotated-corpus/ner_dataset.csv"
if not os.path.exists(path):
    path = "NER dataset.csv"

data = pd.read_csv(path, encoding="latin1")
data["Sentence #"] = data["Sentence #"].ffill()
data = data.dropna(subset=["Word"])

sentences = [list(g["Word"]) for _, g in data.groupby("Sentence #", sort=False)]
gold = [list(g["Tag"]) for _, g in data.groupby("Sentence #", sort=False)]   # only for the final check

print("sentences:", len(sentences), "| words:", len(data))
print(" ".join(sentences[0]))

sentences: 47959 | words: 1048565
Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country .


## ii : Telling a name from an ordinary word

A capital letter is the one clue English gives away for free - names are written with one.
That single clue does most of the work here.

It has one hole. **The first word of every sentence is capitalised** whether it is a name or
not, so *The*, *Police* and *Thousands* all look like names at the start of a sentence.

Rather than writing a list of such words by hand, the text is asked instead. Any word that is
ever written in lower case somewhere in the corpus - *the*, *police*, *thousands* - is an
ordinary word, so a capital at the start of a sentence means nothing. A word that is **never**
seen in lower case - *London*, *Iraq*, *Obama* - really is a name.

This is the only thing taken from the corpus, it is one line, and it uses no labels.

In [2]:
seen_lower = {word.lower() for sentence in sentences for word in sentence if word[0].islower()}

print("ordinary words seen in lower case:", len(seen_lower))
for word in ["The", "Police", "Thousands", "London", "Iraq", "Obama", "Bush"]:
    print(f"  {word:10s} ever written in lower case? {word.lower() in seen_lower}")

ordinary words seen in lower case: 18588
  The        ever written in lower case? True
  Police     ever written in lower case? True
  Thousands  ever written in lower case? True
  London     ever written in lower case? False
  Iraq       ever written in lower case? False
  Obama      ever written in lower case? False
  Bush       ever written in lower case? True


*Bush* shows where the trick fails. A *bush* is also an ordinary plant, so the word does turn up
in lower case, and a sentence that **begins** with *Bush said ...* will have the name thrown
away. The test is only applied to the first word of a sentence, so *Bush* anywhere else is still
picked up normally. Names that double as common words are the price of not writing a list.

## iii : The rules

The sentence is read left to right. At each word the rules below are tried **in order** and the
first one that fits wins, so the strongest clue is always asked first.

**Finding dates** - a date is the one entity with a shape of its own, because it contains digits.

1. A four digit number like `2001` is a **year**.
2. A capital followed by a number, as in `July 21`, is a **date**. Note this catches month names
   without knowing a single month - the shape is enough.
3. A single capitalised word straight after the word `on`, as in `on Friday`, is a **date**.
   In news writing *on* is followed by a day far more often than by anything else.

**Finding names** - a capitalised word starts a name, and the name runs on for as long as the
following words are capitalised too. That is what keeps *Panama City* together as one thing.
At the start of a sentence the lower case test from step ii decides whether it is a name at all.

**Naming what was found** - once a name is found, four questions decide its kind. None of them
looks the name up anywhere; they look at its shape, or at the word on either side of it.

| Question | Answer | Why |
|---|---|---|
| Is it written in block capitals, like `NATO`? | organisation | short forms are almost always bodies |
| Is the word before it `the`? | organisation | we say *the* Taleban, never *the* Bush |
| Is the word before it `in`, `at`, `from` or `near`? | place | you are *in* a place |
| Is the word after it `said`, `says` or `told`? | person | people speak |
| nothing matched | person | a bare name is most often somebody |

Those eight little words - `the`, `in`, `at`, `from`, `near`, `said`, `says`, `told` - are the
only vocabulary in the whole system. They are ordinary English grammar words, not facts about
this dataset.

In [3]:
PLACE_WORDS = {"in", "at", "from", "near"}     # a name after these is usually a place
PERSON_WORDS = {"said", "says", "told"}        # a name before these is usually a person

def is_capital(word):
    return word[0].isupper()

def starts_name(words, i):
    if not is_capital(words[i]):
        return False
    if i == 0:                                  # first word is capitalised no matter what
        return words[i].lower() not in seen_lower
    return True

def tag_sentence(words):
    tags = ["O"] * len(words)
    i = 0
    while i < len(words):
        word = words[i]

        if word.isdigit() and len(word) == 4:                     # 2001
            tags[i] = "B-tim"
            i += 1
        elif is_capital(word) and i + 1 < len(words) and words[i + 1].isdigit():   # July 21
            tags[i], tags[i + 1] = "B-tim", "I-tim"
            i += 2
        elif i > 0 and words[i - 1].lower() == "on" and starts_name(words, i) \
                and not (i + 1 < len(words) and is_capital(words[i + 1])):         # on Friday
            tags[i] = "B-tim"
            i += 1
        elif starts_name(words, i):
            j = i
            while j < len(words) and is_capital(words[j]):        # a name runs on
                j += 1
            before = words[i - 1].lower() if i > 0 else ""
            after = words[j].lower() if j < len(words) else ""
            if all(w.isupper() for w in words[i:j]):
                label = "org"
            elif before == "the":
                label = "org"
            elif before in PLACE_WORDS:
                label = "geo"
            elif after in PERSON_WORDS:
                label = "per"
            else:
                label = "per"
            tags[i] = "B-" + label
            for p in range(i + 1, j):
                tags[p] = "I-" + label
            i = j
        else:
            i += 1
    return tags

The rules now run over the whole corpus. The output uses the same **IOB** form as the dataset -
`B-x` starts an entity of kind `x`, `I-x` continues it, `O` is everything else - so it can be
compared with the real tags later.

In [4]:
predicted = [tag_sentence(words) for words in sentences]

print(" ".join(sentences[554]))
print([(word, tag) for word, tag in zip(sentences[554], predicted[554]) if tag != "O"])

Posada Carriles is wanted in Venezuela for his alleged role in a 1976 Cuban airliner bombing that killed 73 people .
[('Posada', 'B-per'), ('Carriles', 'I-per'), ('Venezuela', 'B-geo'), ('1976', 'B-tim'), ('Cuban', 'B-per')]


## iv : Joining B- and I- tags into whole entities

The tags are one per word, so *Panama City* is still two rows. Chunking walks the sentence once -
a `B-` opens an entity, a matching `I-` carries it on, anything else closes it - and returns
whole entities with their kind. Same function as Part 1, since the output format has not changed.

In [5]:
def entities(words, tags):
    found, part, label = [], [], None
    for word, tag in zip(words, tags):
        if tag.startswith("I-") and tag[2:] == label:
            part.append(word)
        else:
            if part:
                found.append((" ".join(part), label))
            part, label = ([word], tag[2:]) if tag != "O" else ([], None)
    if part:
        found.append((" ".join(part), label))
    return found

for k in [554, 517, 771]:
    print(" ".join(sentences[k]))
    print("   real  :", entities(sentences[k], gold[k]))
    print("   rules :", entities(sentences[k], predicted[k]))

Posada Carriles is wanted in Venezuela for his alleged role in a 1976 Cuban airliner bombing that killed 73 people .
   real  : [('Posada Carriles', 'per'), ('Venezuela', 'geo'), ('1976', 'tim'), ('Cuban', 'gpe')]
   rules : [('Posada Carriles', 'per'), ('Venezuela', 'geo'), ('1976', 'tim'), ('Cuban', 'per')]
On Friday , Iraqi authorities imposed a curfew in Baghdad , because of fears of new violence .
   real  : [('Friday', 'tim'), ('Iraqi', 'gpe'), ('Baghdad', 'geo')]
   rules : [('Friday', 'tim'), ('Iraqi', 'per'), ('Baghdad', 'geo')]
One of the suspected bombers in the failed July 21 attacks on London 's transport system is being extradited from Italy to Britain .
   real  : [('July 21', 'tim'), ('London', 'geo'), ('Italy', 'geo'), ('Britain', 'geo')]
   rules : [('July 21', 'tim'), ('London', 'tim'), ('Italy', 'geo'), ('Britain', 'per')]


The first two sentences come out almost right. *Posada Carriles* is a person, *Venezuela* a
place because of the `in` before it, *1976* a year, *Friday* a date because of the `on` before
it, *Baghdad* a place.

The third shows a rule backfiring. In *attacks on London*, the word *on* is not introducing a
day at all, so *London* is called a date. A rule cannot tell the two uses of *on* apart, because
telling them apart needs to know that London is a city - and that is exactly the knowledge these
rules are not allowed to have.

## v : Checking the rules against the dataset labels

Only now is the `Tag` column opened, and only to mark the answers. `O` is left out of the report
because 85% of the words are `O` and including it flatters any tagger. **Precision** is how often
the rules were right when they claimed an entity, **recall** is how many real entities they found,
and **f1-score** balances the two.

Only the four kinds the rules look for are scored. The dataset also labels nationalities (`gpe`,
words like *Iraqi*) and three rare kinds - `art`, `eve`, `nat` - and no rule was written for any
of them.

In [6]:
from sklearn.metrics import classification_report

y_true = [tag for sentence in gold for tag in sentence]
y_pred = [tag for sentence in predicted for tag in sentence]

print("token accuracy:", round(sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true), 4))
labels = sorted({tag for tag in y_true if tag[2:] in {"geo", "org", "per", "tim"}})
print(classification_report(y_true, y_pred, labels=labels, digits=2, zero_division=0))

token accuracy: 0.8916


              precision    recall  f1-score   support

       B-geo       0.80      0.24      0.36     37644
       B-org       0.38      0.43      0.41     20143
       B-per       0.17      0.72      0.28     16990
       B-tim       0.73      0.26      0.38     20333
       I-geo       0.51      0.14      0.22      7414
       I-org       0.60      0.38      0.47     16784
       I-per       0.48      0.90      0.63     17251
       I-tim       0.79      0.25      0.38      6528

   micro avg       0.38      0.42      0.40    143087
   macro avg       0.56      0.41      0.39    143087
weighted avg       0.58      0.42      0.40    143087



The rules get about **89% of the words right** and a weighted f1 of **0.40** on the entity tags.
Part 1, which was allowed to read the labels, reached 95% and 0.71.

**What the rules do well.** They are good at *finding* entities. Capital letters really do mark
names in English, so most entities are spotted - the recall on `I-per` and `B-org` shows the
spans are being located. Dates work best of the four, because a date is the only kind with a
shape of its own; digits are unmistakable.

**Where they fall down.** Almost all the loss is in saying *what kind* of entity was found.
Nothing about the word *Venezuela* says place and nothing about *Carriles* says person - they are
both just capitalised words. The only evidence available is the neighbouring word, and most of
the time there is no helpful neighbour, so the last rule fires and guesses **person**. That single
fallback is why `per` has the weakest precision and why `geo` and `org` lose so much recall - real
places and organisations are being handed to it.

`gpe` words like *Iraqi* and *Cuban* are pure loss here. They are common in the corpus, and every
one is counted wrong, because a rule has no way to know that *Cuban* is a nationality while
*Carriles* is a surname. Both are simply capitalised words.

**The lesson.** Rules are strong at **detection** and weak at **classification**. Finding where a
name is can be done from the shape of the text. Deciding whether it is a person, a place or an
organisation is knowledge about the world, and the text alone does not contain it. That is the
gap a word list would fill, and the gap that Part 1 filled by counting labels instead.

# Task d : Obtaining the solution - a searchable who / where / when index

The entities are written into four columns - people, organisations, places and dates. Every
sentence is indexed, because with no training step there is nothing to hold back.

In [7]:
columns = {"per": "people", "org": "organisations", "geo": "places", "tim": "dates"}

rows = []
for words, tags in zip(sentences, predicted):
    found = entities(words, tags)
    row = {"sentence": " ".join(words)}
    row.update({name: ", ".join(text for text, label in found if label == short)
                for short, name in columns.items()})
    rows.append(row)

news = pd.DataFrame(rows)
print("sentences indexed:", len(news))
news.head()

sentences indexed: 47959


,sentence,people,organisations,places,dates
0,Thousands of demonstrators have marched throug...,"London, British",,Iraq,
1,Families of soldiers killed in the conflict jo...,"Bush Number One Terrorist, Stop",Bombings,,
2,They marched from the Houses of Parliament to ...,Parliament,Houses,Hyde Park,
3,"Police put the number of marchers at 10,000 wh...",,,,
4,The protest comes on the eve of the annual con...,"Britain, Labor Party, English, Brighton",,,


Counting each column gives the news desk its summary, without anybody reading the sentences.

In [8]:
from collections import Counter

top = {name: Counter(entity for cell in news[name] for entity in cell.split(", ") if entity).most_common(5)
       for name in columns.values()}

pd.DataFrame({name: [f"{entity} ({count})" for entity, count in items] for name, items in top.items()})

,people,organisations,places,dates
0,Iran (1131),U.S. (3100),Iraq (824),2003 (294)
1,Tuesday (1023),United States (1311),Afghanistan (442),2001 (217)
2,Thursday (984),U.N. (556),Baghdad (331),2004 (183)
3,Friday (951),NATO (457),Pakistan (193),2008 (174)
4,Wednesday (904),EU (334),Washington (148),2010 (163)


And a sentence can be looked up by the thing it mentions rather than by the words it uses.

In [9]:
mentions = news["people"] + " | " + news["organisations"] + " | " + news["places"] + " | " + news["dates"]

def search(name):
    return news[mentions.str.contains(name, regex=False)]

hits = search("Iraq")
print("sentences mentioning Iraq:", len(hits))
hits[["sentence", "people", "places", "dates"]].head()

sentences mentioning Iraq: 2685


,sentence,people,places,dates
0,Thousands of demonstrators have marched throug...,"London, British",Iraq,
5,The party is divided over Britain 's participa...,"Britain, British",,
22,Iraqi military officials say tanks and troops ...,"Iraqi, Mosul, Qaida",Iraq,
25,Officials say al Qaida in Iraq fighters have f...,"Qaida, Baghdad","Iraq, Anbar",
57,U.S. Army officials said Wednesday that they w...,"U.S. Army, Wednesday",Iraq,


## What this shows

The index works, and it was built with about thirty lines of rules, no labelled data and no
knowledge of the subject. The counting table is still roughly right about what the corpus is
about, because the common names come up often enough to survive the mistakes.

But the columns are muddled in a way Part 1's were not. The people column is topped by *Iran*
and *Tuesday* - a country and a weekday - because *guess person* is what happens whenever no
neighbouring word helps. A weekday only reaches the dates column when an *on* happens to sit in
front of it. The rules can see that something is a name; they cannot see what sort of name it is.

**When rules like these are worth writing.** They need no labelled data at all, which is the
expensive thing to get. They run instantly, every answer can be traced to one line, and a
mistake is fixed by editing a rule rather than retraining. For a first pass over text nobody has
labelled yet - to find *where* the names are so a person can label them faster - this is often
enough.

**When they are not.** The moment the job needs the *type* of an entity to be right, rules on
their own run out, and either a word list or a labelled corpus has to supply the world knowledge.
Part 1 got that knowledge by counting labels and scored 0.71. These rules have no such source and
score 0.40, and the whole of that gap is the difference between finding a name and knowing what
it is.